In [ ]:
# === 1. パラメータ ===
import random
import datetime
from pathlib import Path

# 解析対象ディレクトリ（このフォルダ配下すべてを再帰的に処理）
TARGET_DIR = Path("./data/欅坂46/")  # 例: r"C:\\Users\\you\\Pictures"

# 出力ベースディレクトリ（各顔IDごとにサブフォルダを作成）
dt_now = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
OUTPUT_DIR = Path(f"./output/{dt_now}")

# 参照する拡張子（小文字）
# 追加/削除: 対象形式を増減します。動画はコーデック依存で読み取れない場合があるためFFmpeg導入推奨。
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
VIDEO_EXTS = [".mp4", ".mov", ".avi", ".mkv", ".webm"]

# 動画フレーム抽出
# NUM_FRAMES_PER_VIDEO: 1本の動画から等間隔で抜き出す枚数
#   大きくすると: 人物のバリエーションを拾いやすく精度↑、計算時間/重複↑
#   小さくすると: 高速だが見逃し↑
NUM_FRAMES_PER_VIDEO = 12
# BUFFER_SECONDS: 先頭/末尾のスキップ秒数（OP/EDやカット繋ぎのノイズ回避）
#   大きくすると: 端のノイズを避けやすいが可用フレームが減る
#   小さくすると: 端のフレームも使う（ノイズ混入リスク↑）
BUFFER_SECONDS = 1.5

# 顔検出/特徴量抽出 (InsightFace)
# INSIGHTFACE_MODEL: 特徴量抽出モデル
#   "buffalo_l": 精度重視（遅い）／ "buffalo_s": 速度重視（精度低下）
INSIGHTFACE_MODEL = "buffalo_l"
# DETECT_THRESH: 顔検出のスコア閾値(0~1)
#   大きくすると: 誤検出↓（見逃し↑）／ 小さくすると: 検出数↑（誤検出↑）
DETECT_THRESH = 0.6
# MIN_FACE_SIZE: 最小顔サイズ（短辺ピクセル）
#   大きくすると: 小さすぎる顔を除外（ノイズ↓、集合写真の遠景は見逃し↑）
#   小さくすると: 遠景も拾う（誤検出やブレの影響↑）
MIN_FACE_SIZE = 80

# 追加の精度オプション
# USE_AUG_FLIP: （簡易）左右反転TTAの有効化スイッチ（この実装では再推論を省略しており効果は限定的）
#   True: 多少頑健化（コスト僅少）／ False: そのまま
USE_AUG_FLIP = True
# QUALITY_USE_BLUR: ぼけ画像の除外を有効化
#   True: ブレ/ピンボケを排除してクラスタを安定化（データ減少）／ False: 品質フィルタ無し
QUALITY_USE_BLUR = True
# BLUR_VAR_MIN: ぼけ判定の閾値（Laplacian分散）
#   大きくすると: 厳しく除外（良質な顔のみ、データ減）／ 小さくすると: 緩く通す（ノイズ混入）
#   目安: 50~150、環境で最適値が変わります
BLUR_VAR_MIN = 80.0
# FACE_MARGIN: 画質判定用ROIの拡張率（bboxを何割広げるか）
#   大きくすると: 顔周辺の情報も含めてブレ判定（過度に大きいと背景影響↑）
#   小さくすると: 純粋に顔中心で判定
FACE_MARGIN = 0.20

# 次元圧縮（PCA + whitening + L2再正規化）
# PCA_WHITEN: 埋め込みの相関を除去しクラスタ分離を改善
#   True: 分離が良くなることが多い（十分なサンプル数が必要）／ False: 元の埋め込みを使用
PCA_WHITEN = True
# PCA_DIM: PCA後の次元（InsightFaceは512次元が多い）
#   大きくすると: 情報保持↑（ノイズも残る）／ 小さくすると: ノイズ除去↑（情報損失の恐れ）
#   目安: 128~256
PCA_DIM = 256

# クラスタリング（DBSCAN推奨）
# CLUSTERING_METHOD:
#   "dbscan": 密度ベース。eps と min_samples で制御。実務で扱いやすい。
#   "hdbscan": 階層的密度。パラメタ感度は低いが実装が重い場合あり。
#   "connected_components": 類似度しきい値で連結成分（大規模では二乗計算で重い）。
CLUSTERING_METHOD = "dbscan"
# DBSCAN_EPS: 近傍半径（cosine距離） ※AUTO_TUNEがTrueなら初期値として使用
#   大きくすると: まとまりやすい（異人マージのリスク↑）
#   小さくすると: 厳密に分割（同一人物が分裂しやすい）
#   目安: 0.30~0.60
DBSCAN_EPS = 0.35
# DBSCAN_MIN_SAMPLES: クラスタと見なす最小サンプル数
#   大きくすると: 単発ノイズを排除（クラスタ成立に多サンプル必要）
#   小さくすると: 小規模クラスタも成立（誤クラスタのリスク↑）
DBSCAN_MIN_SAMPLES = 3
# AUTO_TUNE_DBSCAN_EPS: k-NN距離分布から eps を自動推定
#   True: データ依存で安定／ False: DBSCAN_EPSをそのまま使用
AUTO_TUNE_DBSCAN_EPS = True
# AUTOTUNE_K: k-NNのk（局所密度の評価範囲）
#   大きくすると: 広い近傍でeps推定（eps↑になりやすくマージ傾向）
#   小さくすると: 局所的で厳しめ（eps↓になりやすく分割傾向）
AUTOTUNE_K = 5
# AUTOTUNE_PERCENTILE: k距離の何パーセンタイルを eps に採用するか
#   大きくすると: eps↑（まとまりやすいが混同リスク↑）
#   小さくすると: eps↓（分割しやすいが断片化リスク↑）
AUTOTUNE_PERCENTILE = 75
# AUTOTUNE_MAX_SAMPLES: eps推定に使う最大サンプル数（サブサンプリングで速度と安定性を両立）
#   大きくすると: 安定するが計算重い／ 小さくすると: 高速だが推定がブレやすい
AUTOTUNE_MAX_SAMPLES = 5000

# 事後マージ（デフォルト無効: 以前の挙動を尊重）
# POST_MERGE: 近いクラスタ同士を重心類似度で結合
#   True: 断片化した同一人物を統合（誤マージのリスク）／ False: 何もしない
POST_MERGE = False
# POST_MERGE_COS_DIST_THR: マージ判定のcos距離しきい値（距離=1-類似度）
#   小さくすると: ほぼ同一だけを結合（安全）／ 大きくすると: 緩めに統合（誤マージ↑）
#   目安: 0.25~0.45（類似度0.75~0.55相当）
POST_MERGE_COS_DIST_THR = 0.35

# other（未割当/ノイズ等）の扱い
# SEND_NOISE_TO_OTHER: DBSCAN/HDBSCANのノイズ（label=-1）を id_other/ に送る
#   True 推奨。分類結果に影響させず、コピー段階のみで整理。
SEND_NOISE_TO_OTHER = True
# SEND_SMALL_CLUSTERS_TO_OTHER: 小クラスタ（枚数が少ない）を other へ
#   True: サンプル不足のクラスタを保留に回す／ False: 以前の分類を尊重（デフォルト）
SEND_SMALL_CLUSTERS_TO_OTHER = False
# MIN_CLUSTER_SIZE_FOR_ID: 上記がTrueのとき、これ未満のクラスタは other 扱い
#   大きくすると: 厳密（小規模はotherへ）／ 小さくすると: 小規模でもID付与
MIN_CLUSTER_SIZE_FOR_ID = 2
# COPY_TO_OTHER_IF_HAS_VALID_IDS: 有効IDが付いている画像も other に重複コピーするか
#   True: 「あやしいが有効IDあり」もotherと両方に配置／ False: 有効IDがあればotherへは入れない
COPY_TO_OTHER_IF_HAS_VALID_IDS = False
# NO_FACE_TO_OTHER: そもそも顔が検出されなかったファイルも other に入れるか
#   True: 顔ゼロをotherへ集約／ False: スキップ（どこにも入れない）
NO_FACE_TO_OTHER = False

# ─────────────────────────────────────────────────────────────
# ★ 最小の人物数（下限）ヒント
#   目的: 「最低でも何人分のIDはあるはず」という人間の知識をクラスタリングに反映。
#   使い方: >0 に設定すると、AUTO_TUNE後に「クラスタ数がこの下限を満たすように」
#           DBSCAN では eps を段階的に“小さく”して分割方向にバイアスします
#           （connected_components の場合は類似度しきい値を“高く”して分割方向）。
#   注意: 下限を大きくし過ぎると過分割（同一人物の分裂）が増えます。
MIN_EXPECTED_PERSONS = 0   # 0: 無効（従来通り）。例: 8, 12, 20 など。

# チューニング範囲/ステップ（MIN_EXPECTED_PERSONS>0 のときのみ有効）
# DBSCAN 用: eps の探索範囲と1ステップの縮小幅（cosine距離）
#   EPS_SEARCH_MIN: 下げ止め（これ以下にはしない）※小さいほど分割しやすい
#   EPS_SEARCH_MAX: 上げ止め（上限）※AUTO_TUNEの結果がここを超えることは通常少ない
#   EPS_SEARCH_STEP: 1反復でepsをどれだけ下げるか（小さくすると微調整、反復回数↑）
EPS_SEARCH_MIN  = 0.25
EPS_SEARCH_MAX  = 0.60
EPS_SEARCH_STEP = 0.02

# connected_components 用: 類似度しきい値の探索範囲と1ステップの増分
#   CC_SIM_MIN/MAX: 類似度しきい値の下限/上限（高いほど厳密＝分割方向）
#   CC_SIM_STEP: 1反復の増分（小さくすると微調整、反復回数↑）
CC_SIM_MIN   = 0.55
CC_SIM_MAX   = 0.75
CC_SIM_STEP  = 0.01

# 反復回数の上限（安全装置）
#   大きくすると: 目標達成の可能性↑（遅くなる）／ 小さくすると: 早いが未達の可能性↑
MIN_PERSONS_TUNING_MAX_ITERS = 10
# ─────────────────────────────────────────────────────────────

# その他
# DRY_RUN: Trueならコピーを実行せず計画のみ表示（安全確認用）
#   True: ファイル操作しない／ False: 実際にコピーを実行
DRY_RUN = False
# RANDOM_SEED: 乱数シード（サブサンプリングや一部処理の再現性確保）
#   変えると: 自動推定などのわずかなゆらぎが出る場合あり
RANDOM_SEED = 42

print("Parameters loaded.")

In [ ]:
# === 2. 必要な関数定義 ===
import os, shutil, random
import cv2
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple
from tqdm import tqdm

import insightface
from insightface.app import FaceAnalysis

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from collections import defaultdict, Counter

try:
    import hdbscan  # optional
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

@dataclass
class MediaRecord:
    path: Path
    is_video: bool

def _is_under(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False

def list_media_files(root: Path, image_exts: List[str], video_exts: List[str], exclude_dirs: List[Path] = None) -> List[MediaRecord]:
    if exclude_dirs is None:
        exclude_dirs = []
    recs: List[MediaRecord] = []
    for dirpath, dirnames, filenames in os.walk(root):
        cur_dir = Path(dirpath)
        if any(_is_under(cur_dir, ex) for ex in exclude_dirs):
            dirnames[:] = []
            continue
        for fname in filenames:
            p = cur_dir / fname
            if not p.is_file():
                continue
            ext = p.suffix.lower()
            if ext in image_exts:
                recs.append(MediaRecord(p, is_video=False))
            elif ext in video_exts:
                recs.append(MediaRecord(p, is_video=True))
    return recs

def _expand_bbox(bbox, img_shape, margin: float):
    x1, y1, x2, y2 = bbox
    h, w = img_shape[:2]
    bw = x2 - x1
    bh = y2 - y1
    mx = bw * margin
    my = bh * margin
    nx1 = max(0, int(x1 - mx))
    ny1 = max(0, int(y1 - my))
    nx2 = min(w-1, int(x2 + mx))
    ny2 = min(h-1, int(y2 + my))
    return nx1, ny1, nx2, ny2

def _blur_variance(gray_roi: np.ndarray) -> float:
    return float(cv2.Laplacian(gray_roi, cv2.CV_64F).var())

def sample_frames_from_video(path: Path, num_frames: int, buffer_sec: float) -> List[np.ndarray]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"[WARN] Failed to open video: {path}")
        return []
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cv2.CAP_PROP_FRAME_COUNT)
    if fps <= 0 or total_frames <= 0:
        print(f"[WARN] Invalid video meta: {path}")
        cap.release()
        return []
    duration = total_frames / fps

    start_t = max(0.0, buffer_sec)
    end_t = max(start_t, duration - buffer_sec)

    if num_frames <= 0 or end_t <= start_t:
        times = []
    else:
        if num_frames == 1:
            times = [ (start_t + end_t) / 2.0 ]
        else:
            times = [ start_t + (end_t - start_t) * i / (num_frames - 1) for i in range(num_frames) ]

    frames = []
    for t in times:
        idx = int(round(t * fps))
        idx = np.clip(idx, 0, total_frames - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append(frame)
    cap.release()
    return frames

def load_face_app(model_name: str) -> FaceAnalysis:
    app = FaceAnalysis(name=model_name, providers=['CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    return app

def _face_big_enough(face, min_size: int) -> bool:
    if not hasattr(face, "bbox") or face.bbox is None:
        return True
    x1, y1, x2, y2 = face.bbox.astype(int)
    w = max(0, x2 - x1)
    h = max(0, y2 - y1)
    return min(w, h) >= min_size

def extract_face_embeddings(app: FaceAnalysis, image_bgr: np.ndarray,
                            det_thresh: float, min_face_size: int,
                            use_aug_flip: bool = False,
                            quality_use_blur: bool = False,
                            blur_var_min: float = 80.0,
                            face_margin: float = 0.20) -> List[np.ndarray]:
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces = app.get(img_rgb)
    embs: List[np.ndarray] = []
    for f in faces:
        if (f.det_score is not None and f.det_score < det_thresh):
            continue
        if not _face_big_enough(f, min_face_size):
            continue
        if quality_use_blur and hasattr(f, "bbox") and f.bbox is not None:
            x1, y1, x2, y2 = f.bbox.astype(int)
            ex1, ey1, ex2, ey2 = _expand_bbox((x1, y1, x2, y2), image_bgr.shape, face_margin)
            roi = image_bgr[ey1:ey2, ex1:ex2]
            if roi.size > 0:
                gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
                if _blur_variance(gray) < blur_var_min:
                    continue
        if hasattr(f, "normed_embedding") and f.normed_embedding is not None:
            emb = np.array(f.normed_embedding, dtype=np.float32)
            emb /= (np.linalg.norm(emb) + 1e-12)  # 保険で正規化
            embs.append(emb)
    return embs

def autotune_dbscan_eps(embeddings: np.ndarray, k: int = 5, percentile: int = 75, max_samples: int = 5000) -> float:
    n = embeddings.shape[0]
    if n <= k:
        return 0.5
    idx = np.random.choice(n, size=min(n, max_samples), replace=False)
    X = embeddings[idx]
    nn = NearestNeighbors(n_neighbors=k, metric="cosine")
    nn.fit(X)
    dists, _ = nn.kneighbors(X, return_distance=True)
    kth = np.sort(dists[:, -1])
    eps = float(np.percentile(kth, percentile))
    return eps

def pca_whiten_and_renorm(X: np.ndarray, dim: int, seed: int = 42) -> np.ndarray:
    dim = min(dim, X.shape[1])
    pca = PCA(n_components=dim, whiten=True, random_state=seed)
    Xt = pca.fit_transform(X)
    n = np.linalg.norm(Xt, axis=1, keepdims=True) + 1e-12
    return (Xt / n).astype(np.float32)

def cluster_dbscan(X: np.ndarray) -> np.ndarray:
    eps = DBSCAN_EPS
    if AUTO_TUNE_DBSCAN_EPS:
        eps = autotune_dbscan_eps(X, k=AUTOTUNE_K, percentile=AUTOTUNE_PERCENTILE, max_samples=AUTOTUNE_MAX_SAMPLES)
        print(f"[AUTO-TUNE] DBSCAN eps ~= {eps:.4f} (cosine distance)")
    db = DBSCAN(eps=eps, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine")
    return db.fit_predict(X).astype(int)

def cluster_hdbscan(X: np.ndarray) -> np.ndarray:
    if not HDBSCAN_AVAILABLE:
        raise RuntimeError("HDBSCAN is not installed. Please `uv add hdbscan`")
    clusterer = hdbscan.HDBSCAN(min_cluster_size=max(5, DBSCAN_MIN_SAMPLES+1), metric="euclidean")
    return clusterer.fit_predict(X).astype(int)

def cluster_connected_components(X: np.ndarray, sim_th: float, max_embs: int) -> np.ndarray:
    n = X.shape[0]
    if n > max_embs:
        print(f"[WARN] Too many embeddings for CC ({n}>{max_embs}), falling back to DBSCAN.")
        return cluster_dbscan(X)
    S = np.matmul(X, X.T)              # L2正規化済み前提の cosine 類似度
    A = (S >= sim_th)
    labels = -np.ones(n, dtype=int)
    cid = 0
    visited = np.zeros(n, dtype=bool)
    for i in range(n):
        if visited[i]:
            continue
        stack = [i]
        visited[i] = True
        members = []
        while stack:
            u = stack.pop()
            members.append(u)
            neigh = np.where(A[u])[0]
            for v in neigh:
                if not visited[v]:
                    visited[v] = True
                    stack.append(v)
        for m in members:
            labels[m] = cid
        cid += 1
    return labels

def ensure_unique_path(dst_dir: Path, filename: str) -> Path:
    cand = dst_dir / filename
    if not cand.exists():
        return cand
    stem, ext = os.path.splitext(filename)
    k = 1
    while True:
        cand2 = dst_dir / f"{stem}__{k}{ext}"
        if not cand2.exists():
            return cand2
        k += 1

print("Function definitions ready.")


In [ ]:
# === 3. 実行（埋め込み → 前処理 → クラスタリング → 最小人数チューニング対応） ===
from collections import defaultdict, Counter

assert TARGET_DIR.exists() and TARGET_DIR.is_dir(), f"Directory not found: {TARGET_DIR}"

# 出力配下は除外して再帰探索（自己ループ防止）
media_list = list_media_files(
    TARGET_DIR,
    IMAGE_EXTS,
    VIDEO_EXTS,
    exclude_dirs=[OUTPUT_DIR]
)
print(f"Found {len(media_list)} media files")

face_app = load_face_app(INSIGHTFACE_MODEL)
print("InsightFace loaded.")

all_embeddings: List[np.ndarray] = []
per_file_indices = defaultdict(list)
file_had_embedding = defaultdict(bool)

for m in tqdm(media_list, desc="Extracting embeddings"):
    if m.is_video:
        frames = sample_frames_from_video(m.path, NUM_FRAMES_PER_VIDEO, BUFFER_SECONDS)
        for frm in frames:
            embs = extract_face_embeddings(
                face_app, frm,
                det_thresh=DETECT_THRESH, min_face_size=MIN_FACE_SIZE,
                use_aug_flip=USE_AUG_FLIP,
                quality_use_blur=QUALITY_USE_BLUR, blur_var_min=BLUR_VAR_MIN,
                face_margin=FACE_MARGIN
            )
            if embs:
                file_had_embedding[m.path] = True
            for e in embs:
                per_file_indices[m.path].append(len(all_embeddings))
                all_embeddings.append(e)
    else:
        data = np.fromfile(str(m.path), dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is None:
            print(f"[WARN] Failed to read image: {m.path}")
            continue
        embs = extract_face_embeddings(
            face_app, img,
            det_thresh=DETECT_THRESH, min_face_size=MIN_FACE_SIZE,
            use_aug_flip=USE_AUG_FLIP,
            quality_use_blur=QUALITY_USE_BLUR, blur_var_min=BLUR_VAR_MIN,
            face_margin=FACE_MARGIN
        )
        if embs:
            file_had_embedding[m.path] = True
        for e in embs:
            per_file_indices[m.path].append(len(all_embeddings))
            all_embeddings.append(e)

if len(all_embeddings) == 0:
    print("No embeddings extracted.")
    embeddings = np.zeros((0, 512), dtype=np.float32)
else:
    embeddings = np.stack(all_embeddings).astype(np.float32)
print("Total embeddings:", embeddings.shape)

# 前処理（PCA+whiten は十分なサンプル時のみ）
if PCA_WHITEN and embeddings.shape[0] >= max(50, PCA_DIM+5):
    print(f"Applying PCA+whiten to {embeddings.shape} -> dim={PCA_DIM}")
    X = pca_whiten_and_renorm(embeddings, dim=PCA_DIM, seed=RANDOM_SEED)
else:
    nrm = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12
    X = (embeddings / nrm).astype(np.float32)

def run_dbscan_with_eps(X: np.ndarray, eps: float) -> np.ndarray:
    db = DBSCAN(eps=float(eps), min_samples=DBSCAN_MIN_SAMPLES, metric="cosine")
    return db.fit_predict(X).astype(int)

def count_non_noise(lbls: np.ndarray) -> int:
    return len({int(l) for l in lbls if l != -1})

# ===== クラスタリング =====
if CLUSTERING_METHOD == "dbscan":
    # ベースeps（AUTO_TUNE or 手動）
    if AUTO_TUNE_DBSCAN_EPS and len(X) > 0:
        eps0 = autotune_dbscan_eps(
            X, k=AUTOTUNE_K, percentile=AUTOTUNE_PERCENTILE, max_samples=AUTOTUNE_MAX_SAMPLES
        )
        print(f"[AUTO-TUNE] DBSCAN eps0 ~= {eps0:.4f}")
    else:
        eps0 = DBSCAN_EPS
        print(f"[FIXED] DBSCAN eps0 = {eps0:.4f}")

    # 探索レンジにクリップ
    eps0 = max(EPS_SEARCH_MIN, min(EPS_SEARCH_MAX, eps0))

    labels = run_dbscan_with_eps(X, eps0)

    # --- 最小の人物数（下限）に合わせて分割方向へ微調整 ---
    if MIN_EXPECTED_PERSONS and MIN_EXPECTED_PERSONS > 0 and len(X) > 0:
        cur_eps = eps0
        cur_labels = labels
        cur_k = count_non_noise(cur_labels)
        print(f"[MIN_PERSONS] start: eps={cur_eps:.4f}, clusters={cur_k}, target>={MIN_EXPECTED_PERSONS}")

        iters = 0
        while cur_k < MIN_EXPECTED_PERSONS and cur_eps > EPS_SEARCH_MIN and iters < MIN_PERSONS_TUNING_MAX_ITERS:
            cur_eps = max(EPS_SEARCH_MIN, cur_eps - EPS_SEARCH_STEP)  # 分割方向へ（epsを下げる）
            trial = run_dbscan_with_eps(X, cur_eps)
            k = count_non_noise(trial)
            print(f"[MIN_PERSONS] iter{iters+1}: eps={cur_eps:.4f} -> clusters={k}")
            if k >= cur_k:     # 改善 or 同等なら採用（単調性は保証されない）
                cur_labels = trial
                cur_k = k
            iters += 1

        labels = cur_labels
        print(f"[MIN_PERSONS] final: eps={cur_eps:.4f}, clusters={cur_k}")

elif CLUSTERING_METHOD == "connected_components":
    # 初期しきい値はレンジ中央から開始
    if len(X) > 0:
        sim0 = (CC_SIM_MIN + CC_SIM_MAX) * 0.5
    else:
        sim0 = CC_SIM_MIN
    print(f"[CC] sim0={sim0:.4f} (range {CC_SIM_MIN:.2f}..{CC_SIM_MAX:.2f}, step {CC_SIM_STEP:.3f})")

    def run_cc(X: np.ndarray, sim_th: float) -> np.ndarray:
        return cluster_connected_components(X, sim_th=float(sim_th), max_embs=CC_MAX_EMBS)

    labels = run_cc(X, sim0)

    # --- 最小の人物数（下限）に合わせて分割方向へ微調整（しきい値↑） ---
    if MIN_EXPECTED_PERSONS and MIN_EXPECTED_PERSONS > 0 and len(X) > 0:
        cur_sim = sim0
        cur_labels = labels
        cur_k = count_non_noise(cur_labels)
        print(f"[MIN_PERSONS][CC] start: sim={cur_sim:.4f}, clusters={cur_k}, target>={MIN_EXPECTED_PERSONS}")

        iters = 0
        while cur_k < MIN_EXPECTED_PERSONS and cur_sim < CC_SIM_MAX and iters < MIN_PERSONS_TUNING_MAX_ITERS:
            cur_sim = min(CC_SIM_MAX, cur_sim + CC_SIM_STEP)  # 分割方向へ（閾値を上げる）
            trial = run_cc(X, cur_sim)
            k = count_non_noise(trial)
            print(f"[MIN_PERSONS][CC] iter{iters+1}: sim={cur_sim:.4f} -> clusters={k}")
            if k >= cur_k:
                cur_labels = trial
                cur_k = k
            iters += 1

        labels = cur_labels
        print(f"[MIN_PERSONS][CC] final: sim={cur_sim:.4f}, clusters={cur_k}")

elif CLUSTERING_METHOD == "hdbscan":
    # HDBSCANはクラスタ数を直接制御しにくいので、そのまま実行（下限チューニングは非対応）
    if not HDBSCAN_AVAILABLE:
        raise RuntimeError("HDBSCAN is not installed. Please `uv add hdbscan`")
    labels = cluster_hdbscan(X)
    if MIN_EXPECTED_PERSONS and MIN_EXPECTED_PERSONS > 0:
        print("[MIN_PERSONS][WARN] HDBSCANは下限人数の直接チューニングに非対応です。必要なら DBSCAN/CC を利用してください。")
else:
    raise ValueError("Unknown CLUSTERING_METHOD")

print("Clustering done. Unique raw labels:", sorted(set(labels.tolist())) if len(labels) else [])

# === ID割当（ノイズ=-1はそのまま保持） ===
non_noise = [lab for lab in sorted(set(labels.tolist())) if lab != -1]
lab_to_pid = {lab: i for i, lab in enumerate(non_noise)}  # 非ノイズのみ連番
labels_pid = np.array([lab_to_pid.get(l, -1) for l in labels], dtype=int)

# 各ファイルの ID 集合と other 判定
file_to_ids: Dict[Path, List[int]] = {}
file_to_other: Dict[Path, bool] = {}

# 非ノイズクラスタのサイズ（小クラスタをotherへ送るかの判定用；デフォ無効）
counter_non_noise = Counter([int(l) for l in labels_pid if l != -1])

for path, idxs in per_file_indices.items():
    ids = set()
    other_flag = False
    for i in idxs:
        pid = int(labels_pid[i])
        if pid == -1:
            if SEND_NOISE_TO_OTHER:
                other_flag = True   # ノイズが含まれていた
            continue
        if SEND_SMALL_CLUSTERS_TO_OTHER and counter_non_noise.get(pid, 0) < MIN_CLUSTER_SIZE_FOR_ID:
            other_flag = True
        ids.add(pid)
    if NO_FACE_TO_OTHER and not file_had_embedding.get(path, False):
        other_flag = True

    file_to_ids[path] = sorted(ids)
    file_to_other[path] = other_flag

print("Sample (first 10):")
for idx, (p, ids) in enumerate(file_to_ids.items()):
    if idx >= 10:
        break
    try:
        rel = p.relative_to(TARGET_DIR)
    except Exception:
        rel = p
    print(str(rel), "-> ids:", ids, "| other:", file_to_other[p])

print("Done.")


In [ ]:
# === 4. 出力（id_{n} / id_other） ===
id_other_dir = OUTPUT_DIR / "id_other"

print(f"Output base: {OUTPUT_DIR}")
if not DRY_RUN:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    id_other_dir.mkdir(parents=True, exist_ok=True)

copy_plan: List[Tuple[Path, Path]] = []

for src_path, ids in file_to_ids.items():
    other_flag = file_to_other[src_path]

    # 1) 有効IDがあるなら id_{n} へ
    for person_id in ids:
        id_dir = OUTPUT_DIR / f"id_{person_id}"
        if DRY_RUN:
            dst_path = id_dir / src_path.name
        else:
            id_dir.mkdir(parents=True, exist_ok=True)
            dst_path = ensure_unique_path(id_dir, src_path.name)
        copy_plan.append((src_path, dst_path))

    # 2) other へ入れる条件（ノイズ含み or 小クラスタ扱い or 顔ゼロ(オプション)）
    put_other = False
    if other_flag and (COPY_TO_OTHER_IF_HAS_VALID_IDS or len(ids) == 0):
        put_other = True
    if put_other:
        if DRY_RUN:
            dst_path = id_other_dir / src_path.name
        else:
            id_other_dir.mkdir(parents=True, exist_ok=True)
            dst_path = ensure_unique_path(id_other_dir, src_path.name)
        copy_plan.append((src_path, dst_path))

print("Planned copies (preview up to 30):")
for i, (src, dst) in enumerate(copy_plan[:30]):
    try:
        rel = src.relative_to(TARGET_DIR)
    except Exception:
        rel = src
    print(f"{rel}  ->  {dst}")
if len(copy_plan) > 30:
    print(f"... and {len(copy_plan) - 30} more")

if not DRY_RUN:
    ok, err = 0, 0
    for src, dst in copy_plan:
        try:
            shutil.copy2(src, dst)
            ok += 1
        except Exception as e:
            print(f"[ERROR] Failed to copy: {src} -> {dst} ({e})")
            err += 1
    print(f"Copy finished. ok={ok}, err={err}")
else:
    print("DRY_RUN=True: No files were copied.")
